In [1]:
import uuid
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from qdrant_client.models import Distance, VectorParams

In [2]:
path_parquet = '/Users/tusharrohilla/Desktop/natural_language_search_engine/data/batch_000000.parquet'

In [3]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

COLLECTION_NAME = "semantic_search"

In [4]:
def index_parquet(parquet_file):

    df = pd.read_parquet(parquet_file)

    print(f"Loaded {len(df)} rows")

    client = QdrantClient(
        host="localhost",
        port=6333,
    )

    # Create collection if it doesn't exist
    if not client.collection_exists(COLLECTION_NAME):

        vector_size = len(df["embedding"].iloc[0])

        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(
                size=vector_size,
                distance=Distance.COSINE,
            ),
        )

        print(
            f"Created collection '{COLLECTION_NAME}' "
            f"with vector size {vector_size}"
        )

    points = []

    for _, row in df.iterrows():

        # Convert embedding
        embedding = row["embedding"]

        if hasattr(embedding, "tolist"):
            embedding = embedding.tolist()

        # Convert metadata recursively
        metadata = row["metadata"]

        def convert_numpy(obj):
            if hasattr(obj, "tolist"):
                return obj.tolist()

            if isinstance(obj, dict):
                return {
                    key: convert_numpy(value)
                    for key, value in obj.items()
                }

            if isinstance(obj, list):
                return [
                    convert_numpy(value)
                    for value in obj
                ]

            return obj

        metadata = convert_numpy(metadata)

        point = PointStruct(
            id=str(
                uuid.uuid5(
                    uuid.NAMESPACE_URL,
                    f"amazon-product:{row['parent_asin']}",
                )
            ),
            vector=embedding,
            payload={
                "parent_asin": str(row["parent_asin"]),
                "document": str(row["document"]),
                "metadata": metadata,
            },
        )

        points.append(point)

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
    )

    print(f"Indexed {len(points)} points into '{COLLECTION_NAME}'")

In [5]:
df = pd.read_parquet(path_parquet)

In [6]:
df.columns

Index(['parent_asin', 'document', 'metadata', 'embedding'], dtype='str')

In [7]:
index_parquet(path_parquet)

Loaded 32 rows
Created collection 'semantic_search' with vector size 768
Indexed 32 points into 'semantic_search'


In [8]:
from src.embedding_generator import EmbeddingGenerator

/Users/tusharrohilla/Desktop/natural_language_search_engine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
generator_obj = EmbeddingGenerator()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9559.48it/s]


In [20]:
COLLECTION_NAME

'semantic_search'

In [21]:
embeddings = generator_obj.generate_embeddings(['Hello I want earphones'])

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.28it/s]


In [11]:
embeddings.shape

(1, 768)

In [12]:
client = QdrantClient(
    host="localhost",
    port=6333,
)

In [13]:
type(embeddings[0])

numpy.ndarray

In [14]:
embeddings = embeddings.tolist()

In [15]:
print(type(embeddings))
print(len(embeddings))
print(type(embeddings[0]))
print(len(embeddings[0]) if hasattr(embeddings[0], "__len__") else "scalar")


<class 'list'>
1
<class 'list'>
768


In [22]:
response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=embeddings[0],
    limit=5,
    with_payload=True,
)

In [23]:
results = []

for point in response.points:

    payload = point.payload or {}

    results.append(
        {
            "id": point.id,
            "score": point.score,
            "parent_asin": payload.get(
                "parent_asin"
            ),
            "document": payload.get(
                "document"
            ),
            "metadata": payload.get(
                "metadata"
            ),
        }
    )

In [25]:
results[1]

{'id': '72be81be-a2ba-4cef-9b8b-335ce91e275a',
 'score': 0.74456036,
 'parent_asin': 'B0BFPXH6PD',
 'document': 'Title: Earphones 4 Pack in-Ear Headphones with Microphone, 3.5mm Jack headsets Compatible with Smartphone/Desk Laptop\n\nBrand: N\\A\n\nCategories: Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Earbud Headphones\n\nDescription:\nEarphones 4 Pack in-Ear Headphones with Microphone, 3.5mm Jack headsets Compatible with Smartphone/Desk Laptop',
 'metadata': {'parent_asin': 'B0BFPXH6PD',
  'title': 'Earphones 4 Pack in-Ear Headphones with Microphone, 3.5mm Jack headsets Compatible with Smartphone/Desk Laptop',
  'brand': 'N\\A',
  'categories': ['Electronics',
   'Headphones, Earbuds & Accessories',
   'Headphones & Earbuds',
   'Earbud Headphones'],
  'price': 'None',
  'rating': 4.3,
  'rating_count': 9,
  'image_url': 'https://m.media-amazon.com/images/I/41-fMSiPtQL._AC_.jpg'}}